In [1]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as ply
from scipy.stats import entropy

In [2]:
device = qml.device('default.qubit',wires=1)

In [4]:
@qml.qnode(device)
def rigid_ansatz(weights):
    qml.RX(weights[0],wires=0)
    return qml.state()

@qml.qnode(device)
def flexible_ansatz(weights):
    qml.Rot(weights[0],weights[1],weights[2],wires=0)
    return qml.state()

In [5]:
def overlap(ansatz,num_params):
    overlap=[]
    for _ in range(3000):
        theta1 = np.random.uniform(0,np.pi,num_params)
        theta2 = np.random.uniform(0,np.pi,num_params)
        state1 = ansatz(theta1)
        state2 = ansatz(theta2)
        over = np.abs(np.vdot(state1,state2))**2
        overlap.append(over)
    return overlap

In [7]:
np.random.seed(42)
rigid_overlap = overlap(rigid_ansatz,1)
flexible_overlap = overlap(flexible_ansatz,3)
p_rigid,_ = np.histogram(rigid_overlap, bins=50, range=(0,1), density=True)
p_flexible,_ = np.histogram(flexible_overlap, bins=50, range=(0,1), density=True)
p_haar = np.ones(50)

In [8]:
p_rigid /= np.sum(p_rigid)
p_flexible /= np.sum(p_flexible)
p_haar /= np.sum(p_haar)

In [9]:
kl_rigid = entropy(p_rigid,p_haar)
kl_flexible = entropy(p_flexible,p_haar)

In [10]:
print(f'rigid ansatz has kl divergence as {kl_rigid}')
print(f'flexible ansatz has kl divergence as {kl_flexible}')

rigid ansatz has kl divergence as 0.38134814869263056
flexible ansatz has kl divergence as 0.05294445878217589
